In [ ]:
import pickle
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

# ============================
# IMPORT SHAP IMPLEMENTATIONS
# ============================
from svm_lr_shap import (
    shap_kernel_instance as shap_kernel_classic,
    plot_shap_values as plot_shap_classic,
)
from bert_shap import (
    predict_proba_bert,
    shap_kernel_instance_bert_fast,
    tokenize_with_word_mapping, aggregate_shap_to_words
)

def load_bert(path):
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = TFAutoModelForSequenceClassification.from_pretrained(path)
    return tokenizer, model

bert_tokenizer, bert_model = load_bert("..//MODELS/model_hs_indobert")
BERT_THRESHOLDS = np.array([0.16, 0.64, 0.18, 0.66, 0.12, 0.34])

Some layers from the model checkpoint at ..//MODELS/model_hs_indobert were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at ..//MODELS/model_hs_indobert.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [30]:
import numpy as np

def tokenize_with_word_mapping(sentence, tokenizer):
    print("\n" + "="*60)
    print("1. MEMULAI FUNGSIONALITAS: tokenize_with_word_mapping")
    print("="*60)
    print(f"Kalimat Input Mentah: '{sentence}'")
    
    # Potong berdasarkan spasi
    words = sentence.split()
    print(f"Hasil sentence.split() [words]: {words}")
    
    # Tokenisasi berbasis array kata asli
    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="tf",
        truncation=True,
        padding=False,
    )
    
    # Ambil pecahan token teks asli (misal: 'mema', '##nusia')
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    
    # Ambil id pemetaan kata dari Hugging Face
    word_ids = encoded.word_ids()
    
    print("\n--- Hasil Pemetaan Subword Tokenizer ---")
    print(f"ID Token (input_ids[0]) : {encoded['input_ids'][0].numpy().tolist()}")
    print(f"Pecahan Token (tokens)  : {tokens}")
    print(f"Peta Indeks Kata(word_ids): {word_ids}")
    print("-" * 60)
    
    # Simulasi pelacakan visual sederhana untuk memperjelas keterkaitan token ke kata asli
    print("Pelacakan Hubungan Token -> Kata Asli:")
    for i, (tok, w_id) in enumerate(zip(tokens, word_ids)):
        asal_kata = words[w_id] if w_id is not None else "TOKEN STRUKTURAL (Diabaikan SHAP)"
        print(f"  * Posisi Token ke-{i} [{tok:9}] -> Berasal dari Kata Indeks: {str(w_id):4} ({asal_kata})")
        
    return words, tokens, word_ids


def aggregate_shap_to_words(words, word_ids, idx_features, shap_values):
    print("\n" + "="*60)
    print("2. MEMULAI FUNGSIONALITAS: aggregate_shap_to_words")
    print("="*60)
    print(f"Daftar Kata Asli Target [words]              : {words}")
    print(f"Indeks Posisi Token Kata Riil [idx_features] : {idx_features.tolist() if hasattr(idx_features, 'tolist') else idx_features}")
    print(f"Nilai Mentah SHAP Per Token [shap_values]     : {[round(x, 4) for x in shap_values]}")
    
    # Inisialisasi list kosong sepanjang total kata asli
    word_shap_list = [0.0] * len(words)
    print(f"Inisialisasi Penampung Nilai Kata [word_shap_list]: {word_shap_list}")
    print("-" * 60)
    
    print("Proses Iterasi Akumulasi Nilai SHAP:")
    for shap_idx, token_idx in enumerate(idx_features):
        word_idx = word_ids[token_idx]
        val_shap = shap_values[shap_idx]
        
        print(f"\n  > Iterasi SHAP ke-{shap_idx}: Memeriksa Token Posisi ke-{token_idx}")
        
        if word_idx is not None:
            kata_target = words[word_idx]
            nilai_lama = word_shap_list[word_idx]
            
            # Proses pertambahan/akumulasi dari token ke wadah kata
            word_shap_list[word_idx] += val_shap
            nilai_baru = word_shap_list[word_idx]
            
            print(f"    * Token ini milik kata: '{kata_target}' (Indeks Kata ke-{word_idx})")
            print(f"    * Perhitungan Matematika: {nilai_lama:.4f} + ({val_shap:.4f}) -> Total Sementara: {nilai_baru:.4f}")
        else:
            print(f"    * Token posisi ke-{token_idx} bernilai None (Token struktural terabaikan).")
            
    print("\n" + "-"*60)
    print("Hasil Akhir Penggabungan Tingkat Kata:")
    for w_idx, kata in enumerate(words):
        print(f"  * Kata '{kata:15}' -> Total Akumulasi Nilai SHAP: {word_shap_list[w_idx]:.4f}")
    print("="*60 + "\n")
    
    return word_shap_list

In [31]:
sentence = "tidak memanusiakan manusia"
words,tokens,word_ids = tokenize_with_word_mapping(sentence,bert_tokenizer)


1. MEMULAI FUNGSIONALITAS: tokenize_with_word_mapping
Kalimat Input Mentah: 'tidak memanusiakan manusia'
Hasil sentence.split() [words]: ['tidak', 'memanusiakan', 'manusia']

--- Hasil Pemetaan Subword Tokenizer ---
ID Token (input_ids[0]) : [2, 119, 9651, 437, 32, 666, 3]
Pecahan Token (tokens)  : ['[CLS]', 'tidak', 'meman', '##usia', '##kan', 'manusia', '[SEP]']
Peta Indeks Kata(word_ids): [None, 0, 1, 1, 1, 2, None]
------------------------------------------------------------
Pelacakan Hubungan Token -> Kata Asli:
  * Posisi Token ke-0 [[CLS]    ] -> Berasal dari Kata Indeks: None (TOKEN STRUKTURAL (Diabaikan SHAP))
  * Posisi Token ke-1 [tidak    ] -> Berasal dari Kata Indeks: 0    (tidak)
  * Posisi Token ke-2 [meman    ] -> Berasal dari Kata Indeks: 1    (memanusiakan)
  * Posisi Token ke-3 [##usia   ] -> Berasal dari Kata Indeks: 1    (memanusiakan)
  * Posisi Token ke-4 [##kan    ] -> Berasal dari Kata Indeks: 1    (memanusiakan)
  * Posisi Token ke-5 [manusia  ] -> Berasal da

In [32]:
idx_features = np.array([1, 2, 3, 4, 5])
# Merujuk ke token:   [tidak,   mema,  ##nusia,  ##kan,  manusia]
shap_values_token = [0.0150, 0.0800,  0.1200, 0.0450, -0.0200]
word_shap_values = aggregate_shap_to_words(words, word_ids, idx_features, shap_values_token)
word_shap_values


2. MEMULAI FUNGSIONALITAS: aggregate_shap_to_words
Daftar Kata Asli Target [words]              : ['tidak', 'memanusiakan', 'manusia']
Indeks Posisi Token Kata Riil [idx_features] : [1, 2, 3, 4, 5]
Nilai Mentah SHAP Per Token [shap_values]     : [0.015, 0.08, 0.12, 0.045, -0.02]
Inisialisasi Penampung Nilai Kata [word_shap_list]: [0.0, 0.0, 0.0]
------------------------------------------------------------
Proses Iterasi Akumulasi Nilai SHAP:

  > Iterasi SHAP ke-0: Memeriksa Token Posisi ke-1
    * Token ini milik kata: 'tidak' (Indeks Kata ke-0)
    * Perhitungan Matematika: 0.0000 + (0.0150) -> Total Sementara: 0.0150

  > Iterasi SHAP ke-1: Memeriksa Token Posisi ke-2
    * Token ini milik kata: 'memanusiakan' (Indeks Kata ke-1)
    * Perhitungan Matematika: 0.0000 + (0.0800) -> Total Sementara: 0.0800

  > Iterasi SHAP ke-2: Memeriksa Token Posisi ke-3
    * Token ini milik kata: 'memanusiakan' (Indeks Kata ke-1)
    * Perhitungan Matematika: 0.0800 + (0.1200) -> Total Sementara: 

[0.015, 0.245, -0.02]

In [36]:
def bobot(z,M):
    def factorial(n):
        value = 1
        for i in range(1,n+1):
            value*=i
        return value
    
    return (factorial(z)*factorial(M-z-1))/factorial(M)

bobot(0,2)

0.5

In [40]:
# pelayanan buruk
f_empty = 0.15     # f(Ø)
f_barang = 0.25        # f(x1)
f_buruk = 0.75        # f(x2)
f_barang_buruk = 0.95     # f(x1, x2)

# barang
marjinal_subset_empty = f_barang - f_empty
marjinal_subset_buruk = f_barang_buruk - f_buruk

#  buruk
marjinal_subset_empty2 = f_buruk - f_empty
marjinal_subset_barang = f_barang_buruk - f_barang

barang = (bobot(0,2)*marjinal_subset_empty+bobot(1,2)*marjinal_subset_buruk)
buruk = (bobot(0,2)*marjinal_subset_empty2+bobot(1,2)*marjinal_subset_barang)

f_empty+barang+buruk

0.9499999999999998

In [47]:
B = np.array([[4,2],
              [7,5]])

np.linalg.pinv(B),np.linalg.inv(B),B.T @ np.linalg.inv(B@B.T)

(array([[ 0.83333333, -0.33333333],
        [-1.16666667,  0.66666667]]),
 array([[ 0.83333333, -0.33333333],
        [-1.16666667,  0.66666667]]),
 array([[ 0.83333333, -0.33333333],
        [-1.16666667,  0.66666667]]))